# Hate Speech Detection on Tweets: VADER vs. Supervised Learning

In [1]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import kagglehub
#download vader lexicon
nltk.download('vader_lexicon', quiet=True)

#load dataset (label 1 = hate tweet, label 0 = other)
path = kagglehub.dataset_download("dv1453/twitter-sentiment-analysis-analytics-vidya")
df = pd.read_csv(f"{path}/train_E6oV3lV.csv")


#split data into train (80%) and test (20%) sets
#stratify=df['label'] keeps the same hate/non-hate ratio
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])


Using Colab cache for faster access to the 'twitter-sentiment-analysis-analytics-vidya' dataset.


In [2]:
#initialize vader sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

def get_prediction(text):

    scores = analyzer.polarity_scores(str(text))
    if scores["compound"] <= -0.05:
        return 1 #flag as hate speech
    else:
        return 0


#apply get_prediction to the tweet column
test_df["vader_pred"] = test_df["tweet"].apply(get_prediction)


In [3]:
#evaluation

print("Confusion Matrix:")
print(confusion_matrix(test_df["label"], test_df["vader_pred"]))

print("Classification Report:")
print(classification_report(test_df["label"], test_df["vader_pred"]))

print("Predicted Distribution")
counts = test_df["vader_pred"].value_counts()
print(f"Hate (1): {counts.get(1, 0)}")
print(f"Non-hate (0): {counts.get(0, 0)}")


Confusion Matrix:
[[4908 1037]
 [ 273  175]]
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.83      0.88      5945
           1       0.14      0.39      0.21       448

    accuracy                           0.80      6393
   macro avg       0.55      0.61      0.55      6393
weighted avg       0.89      0.80      0.84      6393

Predicted Distribution
Hate (1): 1212
Non-hate (0): 5181


### VADER Results
Accuracy is 80%, but this doesn't say much. Only about 7% of the tweets are label 1, so a model that always predicts "not hate" would get 93%. What matters is label 1: precision 0.14, recall 0.39, F1 0.21.

VADER marked 1,212 tweets as hateful because of their negative tone, and only 175 of them were really label 1. This makes sense, since VADER measures how negative a tweet sounds, not whether it is racist or sexist. Plenty of negative tweets aren't hateful, and some hateful ones sound neutral.

## Supervised Learning (TF-IDF + Naive Bayes / Logistic Regression)
VADER only looks at tone, so here I train models on the actual labels. The tweets are cleaned with `clean_text`, turned into numbers with a TF-IDF vectorizer, and used to train Naive Bayes and Logistic Regression. Logistic Regression uses `class_weight='balanced'` because label 1 is rare.

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
import re

def clean_text(text):
    text =text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"\d+","", text)
    text = re.sub(r"[^a-z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    text = " ".join([word for word in text.split() if len(word) > 2])
    return text

#apply text cleaning separately
train_cleaned = train_df["tweet"].apply(clean_text)
test_cleaned = test_df["tweet"].apply(clean_text)

vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
X_train = vectorizer.fit_transform(train_cleaned)
X_test = vectorizer.transform(test_cleaned)

y_train = train_df["label"]
y_test = test_df["label"]

### Model 1: Multinomial Naive Bayes
Naive Bayes is a common baseline for text classification. I use it without any class weighting, so it learns from the data as it is. Since the most frequent label is 0 (not hate), it tends to predict "not hate" unless it is quite sure.

In [5]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)

print("Naive Bayes results:")
print(classification_report(y_test, nb_pred))
print(confusion_matrix(y_test, nb_pred))

Naive Bayes results:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      5945
           1       0.90      0.37      0.53       448

    accuracy                           0.95      6393
   macro avg       0.93      0.68      0.75      6393
weighted avg       0.95      0.95      0.94      6393

[[5927   18]
 [ 281  167]]


### Model 2: Logistic Regression
I use `class_weight='balanced'` here, so mistakes on label 1 count more.

In [6]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(class_weight='balanced', max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print("Logistic Regression results:")
print(classification_report(y_test, lr_pred))
print(confusion_matrix(y_test, lr_pred))

Logistic Regression results:
              precision    recall  f1-score   support

           0       0.98      0.93      0.96      5945
           1       0.46      0.78      0.58       448

    accuracy                           0.92      6393
   macro avg       0.72      0.86      0.77      6393
weighted avg       0.95      0.92      0.93      6393

[[5532  413]
 [  97  351]]


### Comparing the models
Both models are tested on the same 6,393 tweets. The important part is label 1 (hateful tweets), where they make different trade-offs.

1. Multinomial Naive Bayes
   * Precision: 0.90 | Recall: 0.37 | F1: 0.53
   * It rarely says "hate" (only 185 tweets), but when it does it is right 90% of the time.
   * It misses 281 of the 448 hateful tweets (63%). Since about 93% of the training tweets are label 0, it leans towards "not hate".

2. Logistic Regression with `class_weight='balanced'`
   * Precision: 0.46 | Recall: 0.78 | F1: 0.58
   * It successfully catches 78% of all hateful tweets, but causes more false alarms, reducing precision to 46%.

Summary: F1 on label 1 goes from 0.21 (VADER) to 0.53 (Naive Bayes) to 0.58 (Logistic Regression). Logistic Regression has the best F1, but which model is better depends on the goal: if missing a hateful tweet is worse than a false alarm, choose Logistic Regression; if false alarms are the bigger problem, choose Naive Bayes.